In [73]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import re
from collections import OrderedDict


In [3]:
df_final_preclinical = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/mapped_preclinical_data_with_mondo_parents.csv")
df_final_clinical = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/mapped_clinical_data_with_mondo_parents.csv")

In [5]:
df_final_clinical.head()

,nct_id,linkbert_aact_mapped_conditions,linkbert_aact_mapped_drugs,Disease Class,drug_term_umls_norm,drug_umls_termid,disease_term_mondo_norm,disease_mondo_termid,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label
0,NCT00000307,alcohol-related disorders|alcoholic cocaine de...,naltrexone,Diseases Category,Naltrexone,C0027360,alcohol-related disorders|alcoholic cocaine de...,MONDO:0021698|-1|MONDO:0005186,-1|MONDO:0005303,-1|drug dependence,MONDO:0021698|-1|MONDO:0005186|MONDO:0005303,alcohol-related disorders|alcoholic cocaine de...
1,NCT00000333,cocaine|cocaine-related disorders,atropine|benzatropine|benztropine|da reuptake ...,Diseases Category,Atropine|Benztropine|Benztropine|da reuptake i...,C0004259|C0005098|C0005098|-1,cocaine|cocaine dependence,-1|MONDO:0005186,MONDO:0005303,drug dependence,-1|MONDO:0005186|MONDO:0005303,cocaine|cocaine dependence|drug dependence
2,NCT00000428,fibromyalgia,amitriptyline|amitriptyline plus fluoxitine|fl...,Neuromuscular Diseases,Amitriptyline|amitriptyline plus fluoxitine|fl...,C0002600|-1|-1|C0016365,fibromyalgia,MONDO:0005546,-1,-1,MONDO:0005546,fibromyalgia
3,NCT00000439,alcohol dependence|bipolar alcoholics|bipolar ...,depacon|sodium valproate|valproate|valproic acid,Psychiatry and Psychology Category,depacon|Valproate Sodium|Valproate|Valproic Acid,-1|C0037567|C0080356|C0042291,alcohol abuse|bipolar alcoholics|bipolar disor...,MONDO:0002046|-1|MONDO:0004985|-1,-1|-1,-1|-1,MONDO:0002046|-1|MONDO:0004985|-1,alcohol abuse|bipolar alcoholics|bipolar disor...
4,NCT00001956,hyperalgesia|oral surgery|pain|removal of thir...,|ketorolac|lidocaine|midazolam|sedative,Neurologic Manifestations,Ketorolac|Lidocaine|Midazolam|sedative,C0073631|C0023660|C0026056|-1,hyperalgesia|oral surgery|obsolete disorder in...,-1|-1|MONDO:0021668|-1,-1,-1,-1|-1|MONDO:0021668|-1,hyperalgesia|oral surgery|obsolete disorder in...


### Collapse similar diseases
- ADAPTED FROM: https://github.com/mims-harvard/PrimeKG/blob/main/knowledge_graph/build_graph.ipynb


In [44]:
def build_disease_nodes(df, col="disease_term_mondo_norm"):
    diseases = (
        df[col]
        .fillna("")
        .str.split("|")
        .explode()
        .str.strip()
    )
    diseases = diseases[diseases != ""]
    diseases = diseases.drop_duplicates().reset_index(drop=True)
    disease_nodes = pd.DataFrame(
        {
            "node_idx": range(len(diseases)),
            "node_name": diseases,
        }
    )
    return disease_nodes


In [45]:

def group_similar_diseases(disease_nodes: pd.DataFrame):
    """
    Groups disease names like:
      - "<root>, <stage>"
      - "<root> type <stage>"
      - "<root> <stage>"
    where <stage> is numeric, roman numeral, single letter, or short alphanumeric (e.g., A1, 1A, A1a).

    Extras:
      - Handles comma-delimited stages: "... , 2"
      - Handles letter+number stages: "type A1", "A2", "1A"
      - Strips trailing 'and'/'or' noise: "... and", "... or"
      - Avoids over-grouping generic roots (post, late, mild to moderate, etc.)
      - Excludes aneuploidy/chromosome terms
      - Treats singular/plural final-word variants as equivalent ("acute seizure" == "acute seizures")
    Returns:
      dict with keys: groups (list of (display_root, indices)),
                      idx2group (node_idx -> display_root),
                      seen (set of grouped indices),
                      excluded (set of excluded indices)
    """

    groups = []
    seen = set()
    idx2group = {}
    no = set()

    # ---------------- Helpers ----------------
    BAD_ROOTS = {
        "mild to moderate",
        "moderate to severe",
        "post",
        "late",
        "early",
        "acute",
        "chronic",
        "type","risk"
    }

    # Noise like "... and", "... or" (with optional punctuation)
    TAIL_NOISE_RE = re.compile(r"\s*(?:and|or)\s*[\.\,\-:;]*\s*$", re.I)

    def strip_trailing_noise(name: str) -> str:
        name = TAIL_NOISE_RE.sub("", str(name).strip())
        name = re.sub(r"\s+", " ", name)
        return name

    def normalize_name_for_comparison(name: str) -> str:
        """
        Normalize a disease/root name for equality comparison:
        - lowercase
        - strip trailing 'and/or' noise
        - remove punctuation
        - collapse whitespace
        - naive singularization of final token (seizures -> seizure)
        """
        name = strip_trailing_noise(name)
        name = re.sub(r"[^\w\s]", "", name.lower()).strip()
        parts = name.split()
        if not parts:
            return ""
        last = parts[-1]
        # naive plural -> singular (avoid 'ss', 'us', 'is')
        if last.endswith("s") and len(last) > 3 and not last.endswith(("ss", "us", "is")):
            parts[-1] = last[:-1]
        return " ".join(parts)

    # Roman numerals
    ROMAN_RE = r"(?:M{0,3}(?:CM|CD|D?C{0,3})(?:XC|XL|L?X{0,3})(?:IX|IV|V?I{0,3}))"

    def isroman(s: str) -> bool:
        return bool(re.fullmatch(ROMAN_RE, s))

    def issingleletter(s: str) -> bool:
        return len(s) == 1 and s.isalpha()

    def is_stage_token(tok: str) -> bool:
        """
        Accept stage/grade tokens:
          - digits: 1, 2, 10
          - roman numerals: I, II, III, IV, ...
          - single letter: A, B, D
          - letter+digits(+optional letter): A1, A2, C10, A1a
          - digits(+optional letter): 1A, 2B
        """
        tok = str(tok).strip()
        if not tok:
            return False
        if tok.isnumeric() or isroman(tok) or issingleletter(tok):
            return True
        if re.fullmatch(r"[A-Za-z]\d{1,3}[A-Za-z]?", tok):
            return True
        if re.fullmatch(r"\d{1,3}[A-Za-z]?", tok):
            return True
        return False

    def norm_root(s: str) -> str:
        # Lower, collapse whitespace, strip trailing commas/hyphens/spaces
        s = re.sub(r"\s+", " ", s.lower()).strip()
        s = re.sub(r"[,\-\s]+$", "", s)
        return s

    # Patterns to extract (root, stage)
    # Supports:
    #   "<root>, <stage>"
    #   "<root> type <stage>"
    #   "<root> <stage>"
    EXTRACT_PATTERNS = [
        re.compile(r"^(?P<root>.+?)\s*,\s*(?:type\s+)?(?P<stage>[A-Za-z0-9]+|"+ROMAN_RE+r")$", re.I),
        re.compile(r"^(?P<root>.+?)\s+(?:type\s+)?(?P<stage>[A-Za-z0-9]+|"+ROMAN_RE+r")$", re.I),
    ]

    def extract_root_and_stage(name: str):
        """
        Returns (root_norm, stage, display_root) or (None, None, None) if not a stage form or blocked.
        display_root preserves original casing/punctuation (trimmed) for nicer printing.
        """
        name = strip_trailing_noise(name)
        for pat in EXTRACT_PATTERNS:
            m = pat.match(name)
            if m:
                root_orig = m.group("root").strip()
                stage = m.group("stage").strip()
                if is_stage_token(stage):
                    root_norm = norm_root(root_orig)
                    if root_norm in BAD_ROOTS:
                        return None, None, None
                    display_root = re.sub(r"[,\-\s]+$", "", root_orig).strip()
                    return root_norm, stage, display_root
        return None, None, None

    # ---------------- Exclusions ----------------
    for i in range(disease_nodes.shape[0]):
        i_name = str(disease_nodes.loc[i, "node_name"])
        i_idx = disease_nodes.loc[i, "node_idx"]
        for w in ["monosomy", "disomy", "trisomy", "trisomy/tetrasomy", "chromosome"]:
            if w.lower() in i_name.lower():
                no.add(i_idx)

    # ---------------- Main grouping ----------------
    for i in range(disease_nodes.shape[0]):
        i_idx = disease_nodes.loc[i, "node_idx"]
        if i_idx in seen or i_idx in no:
            continue

        i_name = str(disease_nodes.loc[i, "node_name"])
        i_root_norm, i_stage, i_display_root = extract_root_and_stage(i_name)

        # Only proceed for stage-like names
        if not i_root_norm:
            continue

        main_root_norm = i_root_norm
        main_display_root = i_display_root  # keep original casing for output

        matches = [i_name]
        matches_idx = [i_idx]
        match_found = False

        for j in range(disease_nodes.shape[0]):
            j_idx = disease_nodes.loc[j, "node_idx"]
            if j_idx in no:
                continue
            j_name = str(disease_nodes.loc[j, "node_name"])
            j_root_norm, j_stage, j_display_root = extract_root_and_stage(j_name)

            # Compare with plural/singular-aware normalization on the roots
            if j_root_norm and normalize_name_for_comparison(j_root_norm) == normalize_name_for_comparison(main_root_norm):
                matches.append(j_name)
                matches_idx.append(j_idx)
                match_found = True
                # Prefer the shortest trimmed display root seen
                if j_display_root and len(j_display_root) < len(main_display_root):
                    main_display_root = j_display_root

        if match_found:
            matches_idx = sorted(set(matches_idx))
            matches = sorted(set(matches))
            if len(matches) <= 1:
                continue

            # Print the group
            print(main_display_root)
            for x in matches:
                print("-  ", x)

            # Mark grouped
            for x in matches_idx:
                seen.add(x)
                idx2group[x] = main_display_root
            groups.append((main_display_root, matches_idx))

    return {
        "groups": groups,
        "idx2group": idx2group,
        "seen": seen,
        "excluded": no,
    }


In [65]:
# ----- Map diseases to their group (if grouped), keep original otherwise -----

def remap_term(term: str) -> str:
    parts = [p.strip() for p in term.split("|") if p.strip()]
    mapped = []
    for p in parts:
        idx = name2idx.get(p)
        if idx is not None and idx in idx2group:
            mapped.append(idx2group[idx])
        else:
            mapped.append(p)
    # de-duplicate while preserving order
    seen_local = set()
    out = []
    for m in mapped:
        if m not in seen_local:
            out.append(m)
            seen_local.add(m)
    return "|".join(out)


In [76]:
MONDO_RE = re.compile(r"^MONDO:\d+$")

# --- assume you already have build_disease_nodes(...) and your group_similar_diseases(...) from your message ---

def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip())

def build_name2group_and_group2id(
    df: pd.DataFrame,
    *,
    raw_col="disease_term_mondo_norm",
    id_col="disease_mondo_termid",
):
    """
    Returns:
      - name2group: {original_term -> grouped_root}
      - group2id:   {grouped_root -> canonical MONDO (parent if present, else first valid, else '-1')}
    """
    # 1) Build nodes + groups
    disease_nodes = build_disease_nodes(df, col=raw_col)
    grp = group_similar_diseases(disease_nodes)
    idx2group = grp["idx2group"]
    idx2name = dict(zip(disease_nodes["node_idx"], disease_nodes["node_name"]))

    # name -> group (or itself if not grouped)
    name2group = {name: idx2group.get(idx, name) for idx, name in idx2name.items()}

    # 2) Compute ONE canonical MONDO per grouped root (simple rules)
    #    - if the root appears as a raw term anywhere with a valid MONDO -> use that (parent)
    #    - else first valid MONDO seen among its children
    #    - else '-1'
    parent_id = {}         # grouped_root -> parent MONDO id (if seen)
    first_valid_child = {} # grouped_root -> first valid child MONDO id

    for _, row in df.iterrows():
        raw_terms = [_norm(x) for x in str(row.get(raw_col, "")).split("|")]
        raw_ids   = [x.strip() for x in str(row.get(id_col, "")).split("|")]

        # pad/truncate if mismatch (keep it simple)
        if len(raw_terms) != len(raw_ids):
            if len(raw_ids) < len(raw_terms):
                raw_ids = raw_ids + ["-1"] * (len(raw_terms) - len(raw_ids))
            else:
                raw_ids = raw_ids[:len(raw_terms)]

        for term, tid in zip(raw_terms, raw_ids):
            if not term:
                continue
            group = _norm(name2group.get(term, term))

            # record parent id if term == group and MONDO valid
            if term == group and MONDO_RE.match(tid or "") and group not in parent_id:
                parent_id[group] = tid

            # record first valid child id if none yet
            if MONDO_RE.match(tid or "") and group not in first_valid_child:
                first_valid_child[group] = tid

    # decide canonical
    group2id = {}
    seen_groups = set(name2group.values())
    for g in seen_groups:
        g_norm = _norm(g)
        if g_norm in parent_id:
            group2id[g_norm] = parent_id[g_norm]
        elif g_norm in first_valid_child:
            group2id[g_norm] = first_valid_child[g_norm]
        else:
            group2id[g_norm] = "-1"

    return name2group, group2id, grp


def apply_grouping_and_ids(
    df: pd.DataFrame,
    name2group: dict,
    group2id: dict,
    *,
    raw_col="disease_term_mondo_norm",
    grouped_col="disease_term_grouped",
    out_id_col="disease_mondo_termid_grouped",
):
    """
    Uses the provided maps to fill:
      - df[grouped_col] with deduped grouped names
      - df[out_id_col] with canonical parent MONDO IDs aligned 1:1 to grouped_col
    """
    def remap_terms_pipe(raw_string: str) -> str:
        parts = [p.strip() for p in str(raw_string).split("|") if p.strip()]
        out, seen_local = [], set()
        for p in parts:
            g = name2group.get(p, p)
            if g not in seen_local:
                out.append(g)
                seen_local.add(g)
        return "|".join(out)

    def ids_for_grouped_pipe(grouped_string: str) -> str:
        parts = [p.strip() for p in str(grouped_string).split("|") if p.strip()]
        ids = [group2id.get(_norm(p), "-1") for p in parts]
        return "|".join(ids)

    df[grouped_col] = df[raw_col].fillna("").apply(remap_terms_pipe)
    df[out_id_col]  = df[grouped_col].apply(ids_for_grouped_pipe)
    return df

In [77]:
def unique_disease_count(df, col):
    return (
        df[col]
        .str.split("|")
        .explode()
        .str.strip()
        .dropna()
        .nunique()
    )



### group clinical

In [115]:
clinical_mondo_diseases = df_final_clinical[['nct_id', 'merged_mondo_label', 'merged_mondo_termid']]
clinical_mondo_diseases.head()

,nct_id,merged_mondo_label,merged_mondo_termid
0,NCT00000307,alcohol-related disorders|alcoholic cocaine de...,MONDO:0021698|-1|MONDO:0005186|MONDO:0005303
1,NCT00000333,cocaine|cocaine dependence|drug dependence,-1|MONDO:0005186|MONDO:0005303
2,NCT00000428,fibromyalgia,MONDO:0005546
3,NCT00000439,alcohol abuse|bipolar alcoholics|bipolar disor...,MONDO:0002046|-1|MONDO:0004985|-1
4,NCT00001956,hyperalgesia|oral surgery|obsolete disorder in...,-1|-1|MONDO:0021668|-1


In [116]:
# 1) Build the maps once (from the dataframe you have)
name2group, group2id, info = build_name2group_and_group2id(
    clinical_mondo_diseases,
    raw_col="merged_mondo_label",
    id_col="merged_mondo_termid",
)

# 2) Apply to create grouped names and aligned parent IDs
clinical_mondo_diseases_grouped = apply_grouping_and_ids(
    clinical_mondo_diseases,
    name2group,
    group2id,
    raw_col="merged_mondo_label",
    grouped_col="disease_term_mondo_parent_clean",
    out_id_col="disease_termid_mondo_parent_clean",
)

complex regional pain syndrome
-   complex regional pain syndrome type 1
-   complex regional pain syndrome type 2
glycogen storage disease
-   glycogen storage disease II
-   glycogen storage disease III
-   glycogen storage disease VII
-   glycogen storage disease XV
Gaucher disease
-   Gaucher disease type I
-   Gaucher disease type II
-   Gaucher disease type III
mucopolysaccharidosis
-   mucopolysaccharidosis type 1
-   mucopolysaccharidosis type 2
-   mucopolysaccharidosis type 3
-   mucopolysaccharidosis type 6
Alzheimer disease
-   Alzheimer disease 2
-   Alzheimer disease 3
-   Alzheimer disease type 1
facioscapulohumeral muscular dystrophy
-   facioscapulohumeral muscular dystrophy 1
-   facioscapulohumeral muscular dystrophy 2
restless legs syndrome, susceptibility to
-   restless legs syndrome, susceptibility to, 2
-   restless legs syndrome, susceptibility to, 6
myotonic dystrophy
-   myotonic dystrophy type 1
-   myotonic dystrophy type 2
brachydactyly
-   brachydactyly t

/sctmp/sdonev/ipykernel_958636/582097319.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[grouped_col] = df[raw_col].fillna("").apply(remap_terms_pipe)
/sctmp/sdonev/ipykernel_958636/582097319.py:104: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[out_id_col]  = df[grouped_col].apply(ids_for_grouped_pipe)


In [117]:
clinical_mondo_diseases_grouped.head()

,nct_id,merged_mondo_label,merged_mondo_termid,disease_term_mondo_parent_clean,disease_termid_mondo_parent_clean
0,NCT00000307,alcohol-related disorders|alcoholic cocaine de...,MONDO:0021698|-1|MONDO:0005186|MONDO:0005303,alcohol-related disorders|alcoholic cocaine de...,MONDO:0021698|-1|MONDO:0005186|MONDO:0005303
1,NCT00000333,cocaine|cocaine dependence|drug dependence,-1|MONDO:0005186|MONDO:0005303,cocaine|cocaine dependence|drug dependence,-1|MONDO:0005186|MONDO:0005303
2,NCT00000428,fibromyalgia,MONDO:0005546,fibromyalgia,MONDO:0005546
3,NCT00000439,alcohol abuse|bipolar alcoholics|bipolar disor...,MONDO:0002046|-1|MONDO:0004985|-1,alcohol abuse|bipolar alcoholics|bipolar disor...,MONDO:0002046|-1|MONDO:0004985|-1
4,NCT00001956,hyperalgesia|oral surgery|obsolete disorder in...,-1|-1|MONDO:0021668|-1,hyperalgesia|oral surgery|obsolete disorder in...,-1|-1|MONDO:0021668|-1


In [118]:
clinical_mondo_diseases_grouped[clinical_mondo_diseases_grouped["merged_mondo_label"].str.contains(r"\bAlzheimer disease type 1\b", case=False, na=False)].head()
clinical_mondo_diseases_grouped[clinical_mondo_diseases_grouped["merged_mondo_label"].str.contains(r"\bAlzheimer disease type 1\b", case=False, na=False)].head().to_csv("AD_merge_mondo_check.csv", index=False)

In [119]:
count_norm = unique_disease_count(clinical_mondo_diseases_grouped, "merged_mondo_label")
count_grouped = unique_disease_count(clinical_mondo_diseases_grouped, "disease_term_mondo_parent_clean")

print(f"Unique diseases in disease_term_mondo_norm: {count_norm}")
print(f"Unique diseases in disease_term_grouped: {count_grouped}")
print(f"Difference: {count_norm - count_grouped}")

Unique diseases in disease_term_mondo_norm: 10864
Unique diseases in disease_term_grouped: 10796
Difference: 68


In [120]:
def check_term_id_length_alignment(df, term_col, id_col, verbose=True):
    """
    Checks that each row has the same number of pipe-separated entries
    in `term_col` and `id_col`.

    Parameters
    ----------
    df : pd.DataFrame
    term_col : str
        Column containing pipe-separated disease names.
    id_col : str
        Column containing pipe-separated MONDO IDs.
    verbose : bool
        If True, prints a short summary.

    Returns
    -------
    pd.DataFrame of mismatched rows with columns:
        ['row_index', 'term_count', 'id_count', term_col, id_col]
    """
    term_counts = df[term_col].fillna("").apply(lambda x: len([p for p in str(x).split("|") if p.strip()]))
    id_counts   = df[id_col].fillna("").apply(lambda x: len([p for p in str(x).split("|") if p.strip()]))

    mismatched_mask = term_counts != id_counts
    mismatched = df.loc[mismatched_mask, [term_col, id_col]].copy()
    mismatched.insert(0, "row_index", mismatched.index)
    mismatched["term_count"] = term_counts[mismatched_mask].values
    mismatched["id_count"] = id_counts[mismatched_mask].values

    if verbose:
        total = len(df)
        n_mismatch = mismatched.shape[0]
        print(f"✅ Total rows: {total}")
        print(f"⚠️  Mismatched rows: {n_mismatch}")
        if n_mismatch > 0:
            print("First few mismatches:")
            print(mismatched.head())

    return mismatched


In [121]:
mismatches = check_term_id_length_alignment(
    clinical_mondo_diseases_grouped,
    term_col="disease_term_mondo_parent_clean",
    id_col="disease_termid_mondo_parent_clean",
)

✅ Total rows: 18609
⚠️  Mismatched rows: 0


In [122]:
df_final_clinical = df_final_clinical.merge(clinical_mondo_diseases_grouped[['nct_id','disease_term_mondo_parent_clean','disease_termid_mondo_parent_clean']], on="nct_id", how="left")

In [123]:
df_final_clinical.head()

,nct_id,linkbert_aact_mapped_conditions,linkbert_aact_mapped_drugs,Disease Class,drug_term_umls_norm,drug_umls_termid,disease_term_mondo_norm,disease_mondo_termid,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label,disease_term_mondo_parent_clean,disease_termid_mondo_parent_clean
0,NCT00000307,alcohol-related disorders|alcoholic cocaine de...,naltrexone,Diseases Category,Naltrexone,C0027360,alcohol-related disorders|alcoholic cocaine de...,MONDO:0021698|-1|MONDO:0005186,-1|MONDO:0005303,-1|drug dependence,MONDO:0021698|-1|MONDO:0005186|MONDO:0005303,alcohol-related disorders|alcoholic cocaine de...,alcohol-related disorders|alcoholic cocaine de...,MONDO:0021698|-1|MONDO:0005186|MONDO:0005303
1,NCT00000333,cocaine|cocaine-related disorders,atropine|benzatropine|benztropine|da reuptake ...,Diseases Category,Atropine|Benztropine|Benztropine|da reuptake i...,C0004259|C0005098|C0005098|-1,cocaine|cocaine dependence,-1|MONDO:0005186,MONDO:0005303,drug dependence,-1|MONDO:0005186|MONDO:0005303,cocaine|cocaine dependence|drug dependence,cocaine|cocaine dependence|drug dependence,-1|MONDO:0005186|MONDO:0005303
2,NCT00000428,fibromyalgia,amitriptyline|amitriptyline plus fluoxitine|fl...,Neuromuscular Diseases,Amitriptyline|amitriptyline plus fluoxitine|fl...,C0002600|-1|-1|C0016365,fibromyalgia,MONDO:0005546,-1,-1,MONDO:0005546,fibromyalgia,fibromyalgia,MONDO:0005546
3,NCT00000439,alcohol dependence|bipolar alcoholics|bipolar ...,depacon|sodium valproate|valproate|valproic acid,Psychiatry and Psychology Category,depacon|Valproate Sodium|Valproate|Valproic Acid,-1|C0037567|C0080356|C0042291,alcohol abuse|bipolar alcoholics|bipolar disor...,MONDO:0002046|-1|MONDO:0004985|-1,-1|-1,-1|-1,MONDO:0002046|-1|MONDO:0004985|-1,alcohol abuse|bipolar alcoholics|bipolar disor...,alcohol abuse|bipolar alcoholics|bipolar disor...,MONDO:0002046|-1|MONDO:0004985|-1
4,NCT00001956,hyperalgesia|oral surgery|pain|removal of thir...,|ketorolac|lidocaine|midazolam|sedative,Neurologic Manifestations,Ketorolac|Lidocaine|Midazolam|sedative,C0073631|C0023660|C0026056|-1,hyperalgesia|oral surgery|obsolete disorder in...,-1|-1|MONDO:0021668|-1,-1,-1,-1|-1|MONDO:0021668|-1,hyperalgesia|oral surgery|obsolete disorder in...,hyperalgesia|oral surgery|obsolete disorder in...,-1|-1|MONDO:0021668|-1


In [124]:
df_final_clinical.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/mapped_clinical_data_with_mondo_parents_mondo_cleaned.csv", index=False)


### group preclinical

In [103]:
df_final_preclinical.shape

(547365, 13)

In [104]:
df_final_preclinical.head()

,PMID,unique_conditions_linkbert_predictions,unique_interventions_linkbert_predictions,linkbert_mapped_conditions,linkbert_mapped_drugs,disease_term_mondo_norm,disease_mondo_termid,drug_term_umls_norm,drug_umls_termid,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label
0,31733831,asthma,isorhynchophylline,asthma,isorhynchophylline,asthma,MONDO:0004979,isorhynchophylline,C0245133,-1,-1,MONDO:0004979,asthma
1,31733833,myocardial infarction,antgomir-1192|mir-1192|agomir-1192,myocardial infarction,antgomir-1192|mir-1192|agomir-1192,myocardial infarction,MONDO:0005068,antgomir-1192|mir-1192|agomir-1192,-1|-1|-1,-1,-1,MONDO:0005068,myocardial infarction
2,31733925,systemic lupus erythematosus,g2|hla-g2,systemic lupus erythematosus,g2|hla-g2,systemic lupus erythematosus,MONDO:0007915,g2|HLA-G2 Isoform,-1|C0967254,-1,-1,MONDO:0007915,systemic lupus erythematosus
3,31733940,cognitive impairment,minocycline,cognitive impairment,minocycline,cognitive disorder,MONDO:0002039,Minocycline,C0026187,-1,-1,MONDO:0002039,cognitive disorder
4,31734027,oxaliplatin-induced peripheral neuropathy|cumu...,tadalafil|phosphodiesterase type 5 inhibitor t...,oxaliplatin-induced peripheral neuropathy|cumu...,tadalafil|phosphodiesterase type 5 inhibitor t...,oxaliplatin-induced peripheral neuropathy|cumu...,-1|-1|MONDO:0003620,Tadalafil|Tadalafil,C1176316|C1176316,-1,-1,-1|-1|MONDO:0003620,oxaliplatin-induced peripheral neuropathy|cumu...


In [105]:
preclinical_mondo_diseases = df_final_preclinical[['PMID', 'merged_mondo_label', 'merged_mondo_termid']]


In [106]:
name2group, group2id, info = build_name2group_and_group2id(
    preclinical_mondo_diseases,
    raw_col="merged_mondo_label",
    id_col="merged_mondo_termid",
)

# 2) Apply to create grouped names and aligned parent IDs
preclinical_mondo_diseases_grouped = apply_grouping_and_ids(
    preclinical_mondo_diseases,
    name2group,
    group2id,
    raw_col="merged_mondo_label",
    grouped_col="disease_term_mondo_parent_clean",
    out_id_col="disease_termid_mondo_parent_clean",
)

retinitis pigmentosa
-   retinitis pigmentosa 1
-   retinitis pigmentosa 12
-   retinitis pigmentosa 2
-   retinitis pigmentosa 27
-   retinitis pigmentosa 36
-   retinitis pigmentosa 4
developmental and epileptic encephalopathy
-   developmental and epileptic encephalopathy 94
-   developmental and epileptic encephalopathy, 13
-   developmental and epileptic encephalopathy, 14
-   developmental and epileptic encephalopathy, 17
-   developmental and epileptic encephalopathy, 39
-   developmental and epileptic encephalopathy, 4
-   developmental and epileptic encephalopathy, 6
-   developmental and epileptic encephalopathy, 7
-   developmental and epileptic encephalopathy, 75
-   developmental and epileptic encephalopathy, 9
i
-   i and r
-   i r
amyotrophic lateral sclerosis
-   amyotrophic lateral sclerosis type 1
-   amyotrophic lateral sclerosis type 10
-   amyotrophic lateral sclerosis type 6
-   amyotrophic lateral sclerosis type 8
-   amyotrophic lateral sclerosis type 9
glycogen

/sctmp/sdonev/ipykernel_958636/582097319.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[grouped_col] = df[raw_col].fillna("").apply(remap_terms_pipe)
/sctmp/sdonev/ipykernel_958636/582097319.py:104: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[out_id_col]  = df[grouped_col].apply(ids_for_grouped_pipe)


In [107]:
count_norm = unique_disease_count(preclinical_mondo_diseases_grouped, "merged_mondo_label")
count_grouped = unique_disease_count(preclinical_mondo_diseases_grouped, "disease_term_mondo_parent_clean")

print(f"Unique diseases in disease_term_mondo_norm: {count_norm}")
print(f"Unique diseases in disease_term_grouped: {count_grouped}")
print(f"Difference: {count_norm - count_grouped}") #446

Unique diseases in disease_term_mondo_norm: 80278
Unique diseases in disease_term_grouped: 79832
Difference: 446


In [109]:
preclinical_mondo_diseases_grouped.head()

,PMID,merged_mondo_label,merged_mondo_termid,disease_term_mondo_parent_clean,disease_termid_mondo_parent_clean
0,31733831,asthma,MONDO:0004979,asthma,MONDO:0004979
1,31733833,myocardial infarction,MONDO:0005068,myocardial infarction,MONDO:0005068
2,31733925,systemic lupus erythematosus,MONDO:0007915,systemic lupus erythematosus,MONDO:0007915
3,31733940,cognitive disorder,MONDO:0002039,cognitive disorder,MONDO:0002039
4,31734027,oxaliplatin-induced peripheral neuropathy|cumu...,-1|-1|MONDO:0003620,oxaliplatin-induced peripheral neuropathy|cumu...,-1|-1|MONDO:0003620


In [110]:
df_final_preclinical = df_final_preclinical.merge(preclinical_mondo_diseases_grouped[['PMID','disease_term_mondo_parent_clean','disease_termid_mondo_parent_clean']], on="PMID", how="left")

In [111]:
df_final_preclinical.head()

,PMID,unique_conditions_linkbert_predictions,unique_interventions_linkbert_predictions,linkbert_mapped_conditions,linkbert_mapped_drugs,disease_term_mondo_norm,disease_mondo_termid,drug_term_umls_norm,drug_umls_termid,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label,disease_term_mondo_parent_clean,disease_termid_mondo_parent_clean
0,31733831,asthma,isorhynchophylline,asthma,isorhynchophylline,asthma,MONDO:0004979,isorhynchophylline,C0245133,-1,-1,MONDO:0004979,asthma,asthma,MONDO:0004979
1,31733833,myocardial infarction,antgomir-1192|mir-1192|agomir-1192,myocardial infarction,antgomir-1192|mir-1192|agomir-1192,myocardial infarction,MONDO:0005068,antgomir-1192|mir-1192|agomir-1192,-1|-1|-1,-1,-1,MONDO:0005068,myocardial infarction,myocardial infarction,MONDO:0005068
2,31733925,systemic lupus erythematosus,g2|hla-g2,systemic lupus erythematosus,g2|hla-g2,systemic lupus erythematosus,MONDO:0007915,g2|HLA-G2 Isoform,-1|C0967254,-1,-1,MONDO:0007915,systemic lupus erythematosus,systemic lupus erythematosus,MONDO:0007915
3,31733940,cognitive impairment,minocycline,cognitive impairment,minocycline,cognitive disorder,MONDO:0002039,Minocycline,C0026187,-1,-1,MONDO:0002039,cognitive disorder,cognitive disorder,MONDO:0002039
4,31734027,oxaliplatin-induced peripheral neuropathy|cumu...,tadalafil|phosphodiesterase type 5 inhibitor t...,oxaliplatin-induced peripheral neuropathy|cumu...,tadalafil|phosphodiesterase type 5 inhibitor t...,oxaliplatin-induced peripheral neuropathy|cumu...,-1|-1|MONDO:0003620,Tadalafil|Tadalafil,C1176316|C1176316,-1,-1,-1|-1|MONDO:0003620,oxaliplatin-induced peripheral neuropathy|cumu...,oxaliplatin-induced peripheral neuropathy|cumu...,-1|-1|MONDO:0003620


In [114]:
df_final_preclinical.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/mapped_preclinical_data_with_mondo_parents_mondo_cleaned.csv", index=False)
